# Practical 9: Semantic Segmentation of Foliar Lesions with U-Net
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit V (Segmentation & Vision Transformers) | **CO:** CO5  
**Problem Statement Context:** SIH26131 — Stage 3: Pixel-Level Lesion Area Estimation & Severity Grading  

---

## 1. U-Net Architecture (Ronneberger et al., 2015)
U-Net comprises a contracting path (encoder) and an expansive path (decoder):
1. **Contracting Path:** Successive blocks of two $3 	imes 3$ convolutions followed by a $2 	imes 2$ MaxPool, halving spatial dimensions while doubling feature channels.
2. **Expansive Path:** Transposed convolutions ($2 	imes 2$ up-conv) doubling spatial dimensions, concatenated with high-resolution feature maps from the contracting path via **skip connections**.
3. **Skip Connections:** Alleviate vanishing gradients and restore fine spatial boundaries lost during pooling.

### Evaluation Metrics:
- **Sørensen-Dice Coefficient:**
  $$	ext{Dice} = rac{2 |X \cap Y|}{|X| + |Y|} = rac{2 \cdot TP}{2 \cdot TP + FP + FN}$$
- **Intersection over Union (IoU / Jaccard):**
  $$	ext{IoU} = rac{|X \cap Y|}{|X \cup Y|} = rac{TP}{TP + FP + FN}$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.dataset import CropSegmentationDataset
from src.segmenter import UNet, compute_segmentation_metrics, calculate_disease_severity, create_segmentation_overlay

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"[✓] Active Device: {device}")

## 2. Load Segmentation Dataset (Image-Mask Pairs)

In [ ]:
seg_img_dir = REPO_ROOT / "data" / "samples" / "segmentation" / "images"
seg_mask_dir = REPO_ROOT / "data" / "samples" / "segmentation" / "masks"

dataset = CropSegmentationDataset(seg_img_dir, seg_mask_dir, target_size=(256, 256))
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"[✓] Loaded {len(dataset)} segmentation image-mask pairs.")
img_tensor, mask_tensor = dataset[0]
print(f"Image tensor: {img_tensor.shape} | Mask tensor: {mask_tensor.shape}")

## 3. Train U-Net on Lesion Masks (3 Epochs)

In [ ]:
model = UNet(in_channels=3, num_classes=1, features=(16, 32, 64, 128)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

model.train()
for epoch in range(3):
    total_loss = 0.0
    for imgs, masks in dataloader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/3] - Segmentation BCE Loss: {total_loss / len(dataloader):.4f}")

## 4. Evaluation: Dice Score & Severity Calculation

In [ ]:
model.eval()
with torch.no_grad():
    sample_pred = model(img_tensor.unsqueeze(0).to(device))

metrics = compute_segmentation_metrics(sample_pred.cpu(), mask_tensor.unsqueeze(0))
print(f"Dice Coefficient : {metrics['dice_score']:.4f}")
print(f"IoU / Jaccard    : {metrics['iou_jaccard']:.4f}")
print(f"Pixel Accuracy   : {metrics['pixel_accuracy'] * 100:.2f}%")

# Severity scoring
pred_bin = (torch.sigmoid(sample_pred) > 0.5).squeeze().cpu().numpy().astype(np.uint8)
severity_pct, stage = calculate_disease_severity(pred_bin)
print(f"Estimated Affected Leaf Surface Area: {severity_pct}% ({stage})")

## 5. Visual Inspection: Ground Truth vs. U-Net Prediction

In [ ]:
raw_img = (img_tensor.permute(1, 2, 0).numpy() * 255.0).astype(np.uint8)
gt_mask = mask_tensor.squeeze().numpy().astype(np.uint8)
overlay = create_segmentation_overlay(raw_img, pred_bin)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(raw_img)
axes[0].set_title("Input Crop Leaf")
axes[0].axis("off")

axes[1].imshow(gt_mask, cmap="gray")
axes[1].set_title("Ground-Truth Mask")
axes[1].axis("off")

axes[2].imshow(pred_bin, cmap="hot")
axes[2].set_title("U-Net Predicted Lesions")
axes[2].axis("off")

axes[3].imshow(overlay)
axes[3].set_title(f"Alpha Overlay ({severity_pct}% Severity)")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 6. Practical Takeaway
- Pixel-level segmentation enables quantifying the precise surface percentage damaged by disease lesions, converting qualitative inspection into an objective numerical severity metric for extension advisories.